# 02 — Data Cleaning

**Kenapa notebook ini ada?** Notebook 01 menemukan tiga masalah. Notebook ini
memperbaikinya **satu per satu**, dengan alasan tertulis di tiap keputusan.

Prinsip yang dipegang: **cleaning hanya memperbaiki yang salah, bukan mengubah data
jadi "cantik"**. Data yang dibuang atau diisi harus bisa dipertanggungjawabkan.

## 0. Persiapan

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

# notebook dijalankan dari folder notebooks/, jadi naik satu level ke root project
BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
print("root project:", BASE)

RAW = BASE / "data" / "raw" / "telco_churn.csv"
CLEAN = BASE / "data" / "clean" / "churn_clean.csv"
CLEAN.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW)
print("data mentah:", df.shape)

root project: C:\Users\53\Documents\data sains\machine learning\project ML Supervised
data mentah: (7043, 21)


## 1. Masalah #1 — `TotalCharges` bertipe teks dan punya 11 nilai kosong

Konversi paksa ke angka dengan `errors="coerce"`: nilai yang tidak bisa dibaca
angka akan menjadi `NaN`, sehingga kita bisa melihat persis berapa yang rusak.

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("jumlah NaN setelah konversi:", int(df["TotalCharges"].isna().sum()))
print()
print("Profil 11 baris tersebut:")
df[df["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges", "Contract"]].head(11)

jumlah NaN setelah konversi: 11

Profil 11 baris tersebut:


,tenure,MonthlyCharges,TotalCharges,Contract
488,0,52.55,NaN,Two year
753,0,20.25,NaN,Two year
936,0,80.85,NaN,Two year
1082,0,25.75,NaN,Two year
1340,0,56.05,NaN,Two year
3331,0,19.85,NaN,Two year
3826,0,25.35,NaN,Two year
4380,0,20.00,NaN,Two year
5218,0,19.70,NaN,One year
6670,0,73.35,NaN,Two year


**Keputusan: isi dengan 0, bukan median.**

Alasannya: 11 baris itu semuanya `tenure = 0` — pelanggan yang baru mendaftar dan
belum pernah ditagih. Total yang pernah dibayar memang **nol**, bukan "rata-rata
pelanggan lain". Mengisi median justru menciptakan data palsu.

Alternatif lain adalah membuang 11 baris itu (0,16% data). Itu juga sah, tapi kita
kehilangan informasi tentang pelanggan paling baru — padahal kelompok itu justru
yang paling berisiko churn.

In [3]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("NaN tersisa:", int(df["TotalCharges"].isna().sum()))
print("TotalCharges minimum:", df["TotalCharges"].min())

NaN tersisa: 0
TotalCharges minimum: 0.0


## 2. Masalah #2 — `customerID` bukan fitur

`customerID` unik untuk 7.043 baris. Kalau dibiarkan, model bisa "menghafal" ID
tertentu sebagai penanda churn. Itu overfitting yang tidak berguna: pelanggan baru
selalu punya ID baru, jadi pola itu tidak bisa dipakai.

In [4]:
print("sebelum drop:", df.shape)
df = df.drop(columns=["customerID"])
print("sesudah drop:", df.shape)

sebelum drop: (7043, 21)
sesudah drop: (7043, 20)


## 3. Masalah #3 — target masih teks

Model klasifikasi butuh target berupa angka. `Yes` → 1, `No` → 0.

In [5]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
print(df["Churn"].value_counts().to_string())
print()
print("proporsi churn: {:.2%}".format(df["Churn"].mean()))

Churn
0    5174
1    1869

proporsi churn: 26.54%


## 4. Efek samping: 22 duplikat muncul setelah `customerID` dibuang

Sebelum `customerID` dibuang, tidak ada satu pun baris duplikat. Setelah dibuang,
muncul duplikat — karena dua pelanggan **berbeda** bisa punya profil yang sama
persis: gender sama, tenure sama, kontrak sama, tagihan sama.

Ini bukan data rusak. Ini konsekuensi wajar dari membuang kolom identitas.

In [6]:
duplikat = df[df.duplicated(keep=False)]
print("jumlah baris duplikat:", int(df.duplicated().sum()))
print("baris yang terlibat  :", len(duplikat))
print()
print(duplikat.sort_values(["tenure", "MonthlyCharges"]).head(6)
      [["gender", "tenure", "Contract", "MonthlyCharges", "Churn"]].to_string())

jumlah baris duplikat: 22
baris yang terlibat  : 42

      gender  tenure        Contract  MonthlyCharges  Churn
542   Female       1  Month-to-month           19.55      0
1491  Female       1  Month-to-month           19.55      0
5170  Female       1  Month-to-month           19.65      0
6774  Female       1  Month-to-month           19.65      0
4817  Female       1  Month-to-month           19.90      0
6706  Female       1  Month-to-month           19.90      0


In [7]:
# apakah ada kelompok duplikat yang targetnya berbeda? (itu tanda data rusak)
kelompok = df.groupby(list(df.columns), dropna=False)["Churn"].nunique()
print("kelompok duplikat dengan target berbeda:", int((kelompok > 1).sum()))

kelompok duplikat dengan target berbeda: 0


**Keputusan: duplikat dibiarkan, tidak dibuang.**

Alasannya: 22 baris itu adalah **22 pelanggan sungguhan** yang kebetulan punya
profil identik. Kalau dibuang, kita menghapus pelanggan nyata dari data dan
membuat proporsi churn menjadi tidak akurat. Target mereka juga seragam, jadi
tidak ada kontradiksi yang perlu diselesaikan.

Catat satu hal jujur: karena ada 22 profil identik, sebagian kecil data uji bisa
"mirip" dengan data latih. Porsinya 0,3% dari total — terlalu kecil untuk
mengubah kesimpulan, tapi tetap dicatat sebagai limitasi di README.

## 5. Verifikasi hasil cleaning

Jangan percaya "sepertinya sudah bersih". Uji dengan `assert` supaya notebook
berhenti dengan error kalau ada yang terlewat.

Perhatikan: yang diuji **bukan** "tidak ada duplikat", karena duplikat di atas
memang sengaja dibiarkan. Yang diuji adalah hal-hal yang kalau salah akan
merusak model.

In [8]:
assert df.isna().sum().sum() == 0, "masih ada nilai kosong"
assert "customerID" not in df.columns, "customerID belum dibuang"
assert set(df["Churn"].unique()) == {0, 1}, "target bukan 0/1"
assert pd.api.types.is_numeric_dtype(df["TotalCharges"]), "TotalCharges masih teks"
assert len(df) == 7043, "jumlah baris berubah — ada data yang hilang"
print("semua pemeriksaan lulus")
print("bentuk akhir:", df.shape)

semua pemeriksaan lulus
bentuk akhir: (7043, 20)


## 6. Simpan hasil cleaning

`data/clean/churn_clean.csv` adalah satu-satunya sumber untuk notebook berikutnya.
Aturan main project ini: **notebook 03 ke atas tidak boleh membaca data mentah**.
Kalau logika cleaning berubah, cukup jalankan ulang notebook ini.

In [9]:
df.to_csv(CLEAN, index=False)
print("disimpan ke:", CLEAN)
print("ukuran:", round(CLEAN.stat().st_size / 1024, 1), "KB")

disimpan ke: C:\Users\53\Documents\data sains\machine learning\project ML Supervised\data\clean\churn_clean.csv
ukuran: 871.6 KB


## 7. Sebelum vs sesudah

| Aspek | Sebelum | Sesudah |
|---|---|---|
| Bentuk | 7.043 × 21 | 7.043 × 20 |
| `TotalCharges` | teks, 11 kosong | angka, 0 kosong |
| `customerID` | ada | dibuang |
| `Churn` | `Yes`/`No` | `1`/`0` |
| Duplikat | 0 | 22 (sengaja dibiarkan) |
| Baris hilang | — | **0** (tidak ada yang dibuang) |

Perhatikan: **tidak ada satu baris pun yang hilang**. Cleaning di sini memperbaiki
tipe data dan membuang kolom yang salah, bukan memangkas data.